# Alerta de Divergência Anormal

Lê o resultado mais recente da reconciliação (`reconciliation.resultado_indice`) e
dispara um e-mail apenas quando a divergência entre KNIME e Databricks ultrapassar
um limiar considerado anormal — não em toda divergência, já que defasagem de minutos
entre as duas execuções é esperada e documentada (ver ADR-07).

**Limiar de anormalidade:** 1% de diferença absoluta no índice-proxy — o dobro da
maior divergência já observada no projeto (0,43%), calibrado para não gerar ruído
em condições normais.

Notebook aditivo: não altera a lógica de `05_reconciliacao`, apenas lê seu resultado
já gravado.

**Entrada:** tabela `poc_b3_modernizacao.reconciliation.resultado_indice`
**Saída:** e-mail (quando divergência > 1%), sem gravação de tabela nova.

In [0]:
%run ../setup/01_utilitarios_pipeline

In [0]:
# widget - modo de execucao (agendado quando disparado pelo Job, reprocessamento_manual por padrao)
dbutils.widgets.dropdown("modo_execucao", "reprocessamento_manual", ["agendado", "reprocessamento_manual"], "Modo de execucao")

In [0]:
# observabilidade - marca inicio da execucao
from datetime import datetime
inicio_execucao = datetime.now()

In [0]:
# imports
from pyspark.sql import functions as F
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

In [0]:
# funcao de envio de alerta por email (HTML)
def enviar_alerta_divergencia(data_referencia, diferenca, indice_knime, indice_databricks, causa_raiz):
    remetente = "bruno.queles.dataeng@gmail.com"
    destinatario = "bruno.queles.dataeng@gmail.com"
    senha = dbutils.secrets.get(scope="b3-secrets", key="gmail-app-password")

    corpo_html = f"""
    <html>
    <body style="font-family: Arial, sans-serif;">
        <h2 style="color: #c0392b;">B3 - Modernização de Dados — Alerta ALTA</h2>
        <p>Divergência anormal detectada na reconciliação entre KNIME e Databricks.</p>
        <table border="1" cellpadding="8" cellspacing="0" style="border-collapse: collapse;">
            <tr><td><b>Data de referência</b></td><td>{data_referencia}</td></tr>
            <tr><td><b>Diferença absoluta</b></td><td>{diferenca}%</td></tr>
            <tr><td><b>Índice KNIME</b></td><td>{indice_knime}%</td></tr>
            <tr><td><b>Índice Databricks</b></td><td>{indice_databricks}%</td></tr>
            <tr><td><b>Causa raiz</b></td><td>{causa_raiz or 'Não determinada automaticamente'}</td></tr>
        </table>
        <p style="color: #888; font-size: 12px;">Enviado automaticamente pelo projeto B3 - Modernização de Dados.</p>
    </body>
    </html>
    """

    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"B3 - Alerta ALTA: Divergência anormal em {data_referencia}"
    msg["From"] = remetente
    msg["To"] = destinatario
    msg.attach(MIMEText(corpo_html, "html"))

    with smtplib.SMTP("smtp.gmail.com", 587) as servidor:
        servidor.starttls()
        servidor.login(remetente, senha)
        servidor.sendmail(remetente, destinatario, msg.as_string())

    print(f"Alerta enviado para {destinatario}")

In [0]:
# execucao principal - checa se ha reconciliacao pendente, avalia divergencia, registra observabilidade
MODO_EXECUCAO = dbutils.widgets.get("modo_execucao")
LIMIAR_DIVERGENCIA_ANORMAL = 1.0

try:
    ultima_execucao_reconciliacao = (spark.table("poc_b3_modernizacao.observability.pipeline_runs")
        .filter(F.col("notebook") == "05_reconciliacao")
        .orderBy(F.col("inicio").desc())
        .limit(1)
        .collect()
    )

    if ultima_execucao_reconciliacao and ultima_execucao_reconciliacao[0]["status"] == "pendente_knime":
        print("Reconciliacao mais recente esta pendente (CSV do KNIME ausente) - avaliacao pulada, sem reavaliar dado antigo.")
        registrar_execucao(
            notebook="06_alerta_divergencia",
            data_referencia=ultima_execucao_reconciliacao[0]["data_referencia"],
            modo_execucao=MODO_EXECUCAO,
            status="sucesso",
            inicio=inicio_execucao,
            fim=datetime.now(),
            mensagem_erro="Avaliacao pulada - reconciliacao mais recente esta pendente_knime, sem dado novo para avaliar.",
        )
    else:
        df_reconciliacao = (spark.table("poc_b3_modernizacao.reconciliation.resultado_indice")
            .orderBy(F.col("data_carga").desc())
            .limit(1)
        )
        resultado = df_reconciliacao.collect()[0]

        data_referencia = resultado["data_referencia"]
        status = resultado["status"]
        diferenca = resultado["diferenca_absoluta"]
        causa_raiz = resultado["causa_raiz"]
        indice_knime = resultado["indice_proxy_pct_knime"]
        indice_databricks = resultado["indice_proxy_pct_databricks"]

        divergencia_anormal = (status == "diverge") and (diferenca is not None) and (diferenca > LIMIAR_DIVERGENCIA_ANORMAL)

        print(f"Data de referencia: {data_referencia}")
        print(f"Status: {status}")
        print(f"Diferenca absoluta: {diferenca}")
        print(f"Divergencia anormal (> {LIMIAR_DIVERGENCIA_ANORMAL}%): {divergencia_anormal}")

        if divergencia_anormal:
            enviar_alerta_divergencia(data_referencia, diferenca, indice_knime, indice_databricks, causa_raiz)
        else:
            print("Divergencia dentro do esperado - nenhum alerta enviado.")

        registrar_execucao(
            notebook="06_alerta_divergencia",
            data_referencia=data_referencia,
            modo_execucao=MODO_EXECUCAO,
            status="sucesso",
            inicio=inicio_execucao,
            fim=datetime.now(),
        )

except Exception as e:
    registrar_execucao(
        notebook="06_alerta_divergencia",
        data_referencia=None,
        modo_execucao=MODO_EXECUCAO,
        status="falha",
        inicio=inicio_execucao,
        fim=datetime.now(),
        mensagem_erro=str(e),
    )
    raise